<div align="center">

<img src="https://unisallevirtual.lasalle.edu.co/pluginfile.php/1/theme_remui/loaderimage/1785720113/La-Salle-color-RGB.png" width="500">

#
# Programa de Ingeniería
# Maestría en Inteligencia Artificial
# Visión por Computadora
## Actividad 6: Proyecto Final – Implementemos un modelo de clasificación
## VisionVial: reconocimiento de señales de tránsito para personas con daltonismo
## Desarrollado por: Diego Nicolás Ávila Moreno y *(integrante 2)*

</div>

# Introducción

El **daltonismo** (deficiencia en la visión del color) es una condición, en la mayoría de los casos hereditaria, en la que uno de los tres tipos de conos de la retina no funciona o funciona de forma alterada. Se estima que afecta a cerca del 8 % de los hombres y al 0,5 % de las mujeres de ascendencia europea. Sus formas más frecuentes son la **protanopía** (conos L, sensibles al rojo) y la **deuteranopía** (conos M, sensibles al verde), que hacen difícil distinguir rojos, verdes, naranjas y amarillos; la **tritanopía** (conos S, sensibles al azul) es mucho menos común.

En la vía, el color tiene significado. En Colombia las **señales reglamentarias** tienen borde rojo, las **preventivas** son rombos amarillos y las **transitorias** (obras) son naranjas. Una persona con daltonismo puede ver la forma de la señal, pero no siempre su categoría, y en condiciones de poca luz o a distancia el problema se agrava.

**Contexto y área de aplicación.** Este proyecto desarrolla **VisionVial**, un prototipo que combina dos piezas:

1. Un **clasificador de imágenes** entrenado y publicado en **Azure AI Custom Vision** que reconoce 10 tipos de señales de tránsito (área de aplicación: **reconocimiento de objetos**).
2. Una **aplicación web** en Streamlit que toma una foto con la cámara del celular, envía la imagen al modelo publicado, muestra la señal reconocida con su categoría y su color escritos en texto, y además **corrige los colores de la imagen** (daltonización) según el tipo de daltonismo del usuario.

El notebook replica el pipeline de la Actividad 4 (clasificador de animales en Custom Vision) y lo amplía con la evaluación del modelo sobre un conjunto de prueba independiente (precision, recall, F1-score, curvas ROC y AUC) y con un experimento que mide si la corrección de color ayuda a recuperar la información que se pierde con el daltonismo.

# Proceso de Programación

Este notebook se divide en las siguientes secciones:

-	**Preparación del dataset:** descarga el GTSRB (*German Traffic Sign Recognition Benchmark*), selecciona las clases equivalentes a señales usadas en Colombia y crea las carpetas `train/` (para Custom Vision) y `test/` (prueba independiente, **no se sube** a Custom Vision).

-	**Análisis descriptivo de las imágenes:** inventario por clase y conjunto, dimensiones, resolución, profundidad de color, intensidad de los píxeles, presencia de colores, **pérdida de color por tipo de daltonismo** y consumo de memoria.

-	**Aplicación de Azure Custom Vision:** documenta con capturas de pantalla la creación del recurso y del proyecto, la carga y etiquetado de las imágenes, el entrenamiento, las iteraciones y la publicación del modelo.

-	**Uso del servicio:** conecta el notebook con el modelo publicado (*Prediction API*) y verifica que responde.

-	**Evaluación del modelo:** predice el conjunto de prueba y calcula precision, recall, F1-score, matriz de confusión, curvas ROC y AUC, errores por categoría y el efecto del umbral de confianza.

-	**Experimento de daltonismo:** clasifica las imágenes de prueba tal como las percibe una persona con cada tipo de daltonismo, sin corrección y con corrección, y compara el desempeño.

-	**Aplicación:** prueba interactiva desde el notebook y descripción de la aplicación web con cámara desarrollada en Streamlit.

# Carga de Librerías

El notebook se desarrolló sobre Colab usando el Drive del autor. Todas las librerías vienen instaladas en Colab, por lo que **no es necesario instalar paquetes adicionales**. El módulo propio `daltonismo.py` (simulación y corrección de color) se carga desde la carpeta de la aplicación, de modo que el notebook y la aplicación web usan exactamente el mismo código.

**Qué hace el chunk.** Importa las librerías del flujo y conecta Google Drive.

| Importación | Etapa | Uso concreto |
| --- | --- | --- |
| `os`, `io`, `sys`, `time`, `subprocess`, `pathlib.Path` | Todas | Rutas, variables de entorno, bytes en memoria, pausas entre llamadas al servicio y ejecución del script de preparación del dataset. |
| `pandas`, `numpy` | Todas | Tablas de inventario, descriptivos, predicciones y métricas; operaciones con las imágenes como matrices. |
| `matplotlib.pyplot` | Descriptivos y evaluación | Gráficos descriptivos, matriz de confusión, curvas ROC y resultados del experimento. |
| `PIL.Image`, `ImageOps` | Todas | Abrir, convertir, recortar y reducir imágenes. |
| `requests` | Servicio | Llamadas HTTP al endpoint de predicción de Custom Vision. |
| `sklearn.metrics`, `label_binarize` | Evaluación | Reporte de clasificación, matriz de confusión, ROC y AUC. |
| `IPython.display` | Todas | Tablas, títulos y capturas de pantalla. |
| `google.colab.drive` | Todas | Monta Google Drive. Fuera de Colab, `EN_COLAB = False` y se usan carpetas locales. |

In [ ]:
## Rutas, variables de entorno, bytes en memoria y procesos
import os
import io
import sys
import time
import subprocess
from pathlib import Path

## Tratamiento de dataframes y matrices
import pandas as pd
import numpy as np

## Gráficos e imágenes
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

## Llamadas HTTP al servicio de predicción de Azure
import requests

## Métricas de evaluación
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             roc_auc_score, accuracy_score, f1_score)
from sklearn.preprocessing import label_binarize

## Visualización en el notebook (la imagen de IPython se renombra para no tapar a PIL.Image)
from IPython.display import display, Markdown, HTML
from IPython.display import Image as IPImage

## Preparación del espacio de trabajo en Google Drive
try:
    from google.colab import drive  # solo existe en Colab
    drive.mount("/content/drive")
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Hiperparámetros de Archivos y Servicio

**Qué hace el chunk.** Centraliza los valores que controlan el notebook. Para cambiar una carpeta, una clase o un parámetro solo se edita esta celda.

**Datos y carpetas**

| Parámetro | Valor | Qué controla |
| --- | --- | --- |
| `REPO` | `.../2.10 Vision por Computadora/visionvial` | Raíz del repositorio clonado en Drive. Fuera de Colab se toma de la variable de entorno `VISIONVIAL_REPO` o de la carpeta superior a `notebooks/`. |
| `CARPETA_DATASET` | `REPO/data/dataset` | Dataset generado por `python vv.py dataset`, con `train/<etiqueta>/`, `test/<etiqueta>/` y `manifest.csv`. |
| `CARPETA_SNIP` | `REPO/assets/entrenamiento` | Pantallazos del entrenamiento importados con `python vv.py capturas` (con su índice `capturas.json`). |
| `CARPETA_SNIP_APP` | `REPO/assets/app` | Pantallazos de la aplicación (`python vv.py capturas --app`). |
| `CARPETA_DESCRIPTIVOS` | `REPO/resultados/descriptivos` | Tablas (CSV) y gráficos (PNG) del análisis descriptivo. |
| `CARPETA_RESULTADOS` | `REPO/resultados/notebook` | Predicciones del endpoint (caché en CSV), métricas y gráficos de la evaluación hecha desde el notebook. |

**Clases y colores**

Las clases, categorías y colores se toman de `visionvial/config.py`, la misma fuente que usan la línea de comandos y la aplicación: `CATEGORIA` asigna a cada etiqueta su categoría según el manual de señalización de Colombia y el color que la identifica; `COLOR_CLASE` y `COLOR_CATEGORIA` fijan los colores de los gráficos.

**Límites de Custom Vision y del análisis**

| Parámetro | Valor | Qué controla |
| --- | --- | --- |
| `LADO_MINIMO_CV` | `256` px | Lado corto recomendado por Custom Vision para entrenar. |
| `MAX_MB_PREDICCION` | `4` MB | Peso máximo de una imagen enviada a predicción. |
| `LADO_IMAGEN_MEDIA` | `64` px | Resolución de la imagen promedio por clase. |
| `LADO_DALTONISMO` | `128` px | Resolución a la que se calcula la pérdida de color por daltonismo (para que el cálculo sea rápido). |

In [ ]:
# Raíz del repositorio (clonado en Drive en Colab; en local, la carpeta superior a notebooks/)
REPO = Path("/content/drive/MyDrive/Maestria/Maestria IA/2. Clases/2.10 Vision por Computadora/visionvial") if EN_COLAB \
       else Path(os.environ.get("VISIONVIAL_REPO", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

# Paquete del proyecto: clases, daltonismo e índice de capturas (lo mismo que usan vv.py y app.py)
sys.path.insert(0, str(REPO))
from visionvial import config
from visionvial import capturas as cap
from visionvial import daltonismo as dal

CARPETA_DATASET = REPO / "data" / "dataset"
CARPETA_SNIP = REPO / "assets" / "entrenamiento"
CARPETA_SNIP_APP = REPO / "assets" / "app"
CARPETA_DESCRIPTIVOS = REPO / "resultados" / "descriptivos"
CARPETA_RESULTADOS = REPO / "resultados" / "notebook"
for carpeta in (CARPETA_DESCRIPTIVOS, CARPETA_RESULTADOS):
    carpeta.mkdir(parents=True, exist_ok=True)

EXTENSIONES_IMG = (".jpg", ".jpeg", ".png", ".bmp", ".ppm")

# Etiqueta -> (categoría en Colombia, color que la identifica)
CATEGORIA = config.CATEGORIA

# Límites de Custom Vision
LADO_MINIMO_CV = 256        # lado más corto recomendado para entrenar (px)
MAX_MB_PREDICCION = 4       # peso máximo de una imagen enviada a predicción (MB)

# Análisis
LADO_IMAGEN_MEDIA = 64
LADO_DALTONISMO = 128

# Gráficos
COLOR_CLASE = config.COLOR_CLASE
COLOR_CATEGORIA = config.COLOR_CATEGORIA

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})

TIPOS = list(dal.TIPOS)


def guardar(df, nombre, carpeta=CARPETA_DESCRIPTIVOS):
    df.to_csv(carpeta / f"{nombre}.csv")
    return df


print(f"Repositorio: {REPO}")
print("Tipos de daltonismo:", ", ".join(dal.TIPOS.values()))

# Preparación del Dataset

**Fuente.** Se usa el **GTSRB** (*German Traffic Sign Recognition Benchmark*, Stallkamp et al., 2012), un conjunto público de más de 50.000 recortes de señales de tránsito en 43 clases, tomados de video en condiciones reales (distintas distancias, iluminación, desenfoque y oclusiones). Se eligió porque está etiquetado, es abierto y sus señales **reglamentarias** (PARE, ceda el paso, velocidad máxima, prohibido adelantar, no entre) son prácticamente iguales a las colombianas, que siguen la Convención de Viena.

**Selección de clases.** De las 43 clases se toman las que tienen equivalente en Colombia y se agrupan en 10 etiquetas:

| Etiqueta | Clases GTSRB | Categoría (Colombia) | Color |
| --- | --- | --- | --- |
| `velocidad_maxima` | 1, 2, 3, 4, 5, 7, 8 (30 a 120 km/h) | Reglamentaria | Rojo |
| `prohibido_adelantar` | 9 | Reglamentaria | Rojo |
| `ceda_el_paso` | 13 | Reglamentaria | Rojo |
| `pare` | 14 | Reglamentaria | Rojo |
| `no_entre` | 17 | Reglamentaria | Rojo |
| `curva_peligrosa` | 19, 20, 21 | Preventiva | Amarillo |
| `semaforo_adelante` | 26 | Preventiva | Amarillo |
| `peatones` | 27 | Preventiva | Amarillo |
| `zona_escolar` | 28 | Preventiva | Amarillo |
| `obras` | 25 | Transitoria | Naranja |

**Limitación conocida.** En Alemania las señales preventivas son triángulos blancos con borde rojo, mientras que en Colombia son rombos amarillos. El script permite sumar fotos propias (`--extra`) para acercar el dataset a la señalización colombiana.

**Qué hace el chunk.** Si el dataset no existe, ejecuta `python vv.py dataset` (módulo `visionvial/dataset.py`), que:
1. Descarga y descomprime GTSRB (~300 MB).
2. Toma **150 imágenes por etiqueta para entrenamiento** de la partición oficial de entrenamiento. GTSRB guarda unas 30 tomas casi idénticas de cada señal física (*tracks*), por lo que la selección recorre los tracks de forma alternada para maximizar la diversidad y evitar duplicados.
3. Toma **50 imágenes por etiqueta para prueba** de la partición oficial de prueba, que viene de tracks distintos: así el conjunto de prueba no comparte señales físicas con el de entrenamiento.
4. Descarta originales con lado corto menor a 32 px y amplía el resto a 256 px en el lado corto (recomendación de Custom Vision).
5. Escribe `manifest.csv` con el origen y el tamaño original de cada imagen.

Después lee `manifest.csv`.

In [ ]:
if not (CARPETA_DATASET / "train").exists():
    print("Generando el dataset con `python vv.py dataset` (descarga GTSRB, ~300 MB)...")
    subprocess.run([sys.executable, "vv.py", "dataset"], cwd=REPO, check=True)
else:
    print(f"Dataset encontrado en: {CARPETA_DATASET}")

manifest = pd.read_csv(CARPETA_DATASET / "manifest.csv")
# El manifest puede haberse generado en Windows (rutas con \\): se toma solo el nombre del archivo
manifest["archivo"] = manifest["archivo"].str.replace("\\", "/", regex=False).str.split("/").str[-1]
manifest = manifest.rename(columns={"etiqueta": "clase"})
print(f"manifest.csv: {len(manifest)} registros")

# Descriptivo de Imágenes

**Contexto en el desarrollo del modelo.** Antes de entrenar conviene conocer los datos: cuántas imágenes hay por clase y conjunto, qué dimensiones tienen, cómo es su color y cuánta memoria ocupan. En este proyecto el color tiene un papel doble: es lo que permite al modelo (y a las personas) reconocer la categoría de la señal, y es justamente lo que se pierde con el daltonismo. Por eso, además de los descriptivos de la Actividad 4, se mide **qué colores están presentes** en cada clase y **cuánta información de color se pierde** con cada tipo de daltonismo.

Se analizan **todas las imágenes** de los dos conjuntos (entrenamiento y prueba). Las tablas se guardan como CSV y los gráficos como PNG en `CARPETA_DESCRIPTIVOS`.

## Inventario de Imágenes

**Qué hace el chunk.** Recorre `train/` y `test/` y construye la tabla `datos`, con una fila por imagen: nombre, conjunto (`split`), clase, categoría, color de la señal, ruta y extensión. Le agrega del `manifest.csv` el origen (GTSRB o foto propia), la clase original de GTSRB y el tamaño original antes de la ampliación.

Muestra tres tablas:
- `tabla_clases`: imágenes por clase y conjunto, con el porcentaje que representa cada clase;
- `tabla_categorias`: imágenes por categoría (reglamentaria, preventiva y transitoria);
- `tabla_composicion_gtsrb`: de qué clases de GTSRB se compone cada etiqueta en entrenamiento.

**Salida.** *(Completar con los resultados al ejecutar.)* Revisar que las 10 clases estén balanceadas (150 en entrenamiento y 50 en prueba si no se descartaron imágenes) y que las categorías no lo estén: hay 5 reglamentarias, 4 preventivas y 1 transitoria, lo que debe tenerse en cuenta al leer las métricas por categoría.

In [ ]:
def inventario(carpeta_dataset):
    """Una fila por imagen; la subcarpeta es la clase y la carpeta superior el conjunto."""
    filas, ignorados = [], 0
    for split in ("train", "test"):
        carpeta = carpeta_dataset / split
        if not carpeta.exists():
            print(f"No existe la carpeta {carpeta}")
            continue
        for sub in sorted(p for p in carpeta.iterdir() if p.is_dir()):
            for f in sorted(sub.rglob("*")):
                if not f.is_file():
                    continue
                if f.suffix.lower() in EXTENSIONES_IMG:
                    filas.append({"archivo": f.name, "split": split, "clase": sub.name,
                                  "ruta": str(f), "extension": f.suffix.lower()})
                else:
                    ignorados += 1
    if ignorados:
        print(f"Archivos ignorados (no son imágenes): {ignorados}")
    return pd.DataFrame(filas)


datos = inventario(CARPETA_DATASET)
clases = sorted(datos["clase"].unique())
datos["categoria"] = datos["clase"].map(lambda c: CATEGORIA.get(c, ("Sin categoría", "-"))[0])
datos["color_senal"] = datos["clase"].map(lambda c: CATEGORIA.get(c, ("Sin categoría", "-"))[1])
datos = datos.merge(manifest[["split", "clase", "archivo", "fuente", "class_id_gtsrb",
                              "ancho_original", "alto_original"]],
                    on=["split", "clase", "archivo"], how="left")

tabla_clases = (datos.groupby(["categoria", "clase"])["split"].value_counts()
                     .unstack(fill_value=0).reindex(columns=["train", "test"], fill_value=0))
tabla_clases["total"] = tabla_clases.sum(axis=1)
for col in ("train", "test"):
    tabla_clases[f"%_{col}"] = (100 * tabla_clases[col] / tabla_clases[col].sum()).round(1)
tabla_clases.loc[("total", ""), :] = tabla_clases.sum(numeric_only=True).round(1)

tabla_categorias = pd.crosstab(datos["categoria"], datos["split"], margins=True, margins_name="total")
tabla_composicion = pd.crosstab(datos.loc[datos["split"] == "train", "clase"],
                                datos.loc[datos["split"] == "train", "class_id_gtsrb"].fillna(-1).astype(int)
                                    .replace(-1, "propia"))
tabla_composicion.columns.name = "clase GTSRB"

print(f"Imágenes encontradas: {len(datos)} en {len(clases)} clases")
print(f"  Entrenamiento (se suben a Custom Vision): {(datos['split'] == 'train').sum()}")
print(f"  Prueba (no se suben):                     {(datos['split'] == 'test').sum()}")
display(guardar(tabla_clases, "tabla_clases"))
display(guardar(tabla_categorias, "tabla_categorias"))
print("Composición de cada etiqueta por clase original de GTSRB (entrenamiento)")
display(guardar(tabla_composicion, "tabla_composicion_gtsrb"))

## Dimensiones

**Qué hace el chunk.** Lee de la cabecera de cada imagen (sin decodificar los píxeles) el ancho, el alto, el modo de color, el formato y el peso, y calcula megapíxeles, relación de aspecto, orientación, lado corto, si cumple el lado mínimo de Custom Vision, peso en disco y memoria al decodificarla. Además marca `ampliada` cuando el original de GTSRB era más pequeño y el script lo amplió.

El detalle por imagen se guarda en `detalle_por_imagen.csv`.

In [ ]:
BYTES_POR_PIXEL = {"1": 0.125, "L": 1, "P": 1, "LA": 2, "I;16": 2, "RGB": 3,
                   "YCbCr": 3, "RGBA": 4, "CMYK": 4, "I": 4, "F": 4}


def leer_info(ruta):
    """Lee solo la cabecera de la imagen (rápido): no decodifica los píxeles."""
    ruta = Path(ruta)
    try:
        with Image.open(ruta) as img:
            ancho, alto = img.size
            modo, formato = img.mode, img.format
        return pd.Series({"ancho_px": ancho, "alto_px": alto, "modo": modo, "formato": formato,
                          "disco_bytes": ruta.stat().st_size})
    except Exception as e:
        print(f"No se pudo leer {ruta.name}: {e}")
        return pd.Series({"ancho_px": np.nan, "alto_px": np.nan, "modo": None, "formato": None,
                          "disco_bytes": np.nan})


print("Leyendo cabeceras de las imágenes...")
datos = pd.concat([datos, datos["ruta"].apply(leer_info)], axis=1)
datos = datos.dropna(subset=["ancho_px"])

datos["ancho_original"] = datos["ancho_original"].fillna(datos["ancho_px"])
datos["alto_original"] = datos["alto_original"].fillna(datos["alto_px"])
datos["ampliada"] = datos["ancho_original"] < datos["ancho_px"]
datos["megapixeles"] = datos["ancho_px"] * datos["alto_px"] / 1e6
datos["aspecto"] = datos["ancho_px"] / datos["alto_px"]
datos["orientacion"] = np.select([datos["aspecto"] > 1.02, datos["aspecto"] < 0.98],
                                 ["horizontal", "vertical"], default="cuadrada")
datos["lado_corto"] = datos[["ancho_px", "alto_px"]].min(axis=1)
datos["lado_corto_original"] = datos[["ancho_original", "alto_original"]].min(axis=1)
datos["cumple_lado_minimo"] = datos["lado_corto"] >= LADO_MINIMO_CV
datos["disco_mb"] = datos["disco_bytes"] / 1e6
datos["ram_mb"] = (datos["ancho_px"] * datos["alto_px"]
                   * datos["modo"].map(BYTES_POR_PIXEL).fillna(4) / 1e6)

guardar(datos, "detalle_por_imagen")
print("Listo.")

## Dimensión de Imágenes

**Qué hace el chunk.** Resume por clase las dimensiones **originales** (las de GTSRB, que muestran con cuánto detalle se capturó la señal) y las de los archivos **cargados a Custom Vision** (después de la ampliación).

- `tabla_dimensiones_originales`: ancho y alto mínimo, mediano y máximo de los recortes originales, y cuántos estaban por debajo de 256 px.
- `tabla_dimensiones_cargadas`: dimensiones de los archivos generados, cuántas imágenes se ampliaron y cuántas cumplen el lado mínimo.
- `tabla_modo_formato`: combinación de modo de color y formato.

**Salida.** *(Completar con los resultados al ejecutar.)* En GTSRB los recortes suelen ser pequeños (entre unos 30 y 250 px de lado) y casi cuadrados; se espera que la mayoría haya sido ampliada. Esto significa que, aunque los archivos cumplen el lado mínimo, **el detalle real es el del original**: conviene mencionarlo al interpretar los errores del modelo.

In [ ]:
def resumen_dimensiones(df, ancho, alto, lado):
    return df.groupby("clase").agg(
        imagenes=("archivo", "count"),
        ancho_min=(ancho, "min"), ancho_mediana=(ancho, "median"), ancho_max=(ancho, "max"),
        alto_min=(alto, "min"), alto_mediana=(alto, "median"), alto_max=(alto, "max"),
        bajo_lado_minimo=(lado, lambda s: int((s < LADO_MINIMO_CV).sum())),
    ).round(1)


dim_originales = resumen_dimensiones(datos, "ancho_original", "alto_original", "lado_corto_original")
dim_cargadas = resumen_dimensiones(datos, "ancho_px", "alto_px", "lado_corto")
dim_cargadas["ampliadas"] = datos.groupby("clase")["ampliada"].sum()
dim_cargadas["aspecto_mediana"] = datos.groupby("clase")["aspecto"].median().round(2)

modos = pd.crosstab(datos["clase"], datos["modo"] + " / " + datos["formato"])
modos.columns.name = "modo / formato"

print("Dimensiones ORIGINALES de los recortes de GTSRB (ancho × alto, px)")
display(guardar(dim_originales, "tabla_dimensiones_originales"))
print(f"Dimensiones de los archivos CARGADOS a Custom Vision (lado corto mínimo {LADO_MINIMO_CV} px)")
display(guardar(dim_cargadas, "tabla_dimensiones_cargadas"))
print("Modo de color / formato")
display(guardar(modos, "tabla_modo_formato"))

## Pixeles y Colores

**Contexto en el desarrollo del modelo.** Este bloque **abre cada imagen completa** y analiza su contenido. Responde cuatro preguntas:

- **Profundidad de color:** ¿cuántos canales y bits por píxel tienen las imágenes, y alguna es en realidad gris?
- **Intensidad:** ¿qué tan brillantes y contrastadas son? GTSRB incluye muchas tomas con poca luz o a contraluz.
- **Presencia de colores:** ¿qué porcentaje de cada imagen es rojo, amarillo/naranja, verde o azul? Esto permite comprobar si el color que define cada categoría está realmente presente en las imágenes.
- **Pérdida de color por daltonismo:** ¿cuánto cambia la imagen cuando se simula cada tipo de daltonismo, y cuánto cambia lo que la persona percibe cuando se aplica la corrección?

### Parámetros y Funciones Iniciales

**Qué hace el chunk.** Define los parámetros y la función de análisis, decodifica todas las imágenes y guarda una fila de resultados por imagen.

| Parámetro | Valor | Qué controla |
| --- | --- | --- |
| `UMBRAL_NEGRO`, `UMBRAL_BLANCO` | `10`, `245` | Intensidad por debajo o por encima de la cual un píxel se cuenta como negro o blanco saturado. |
| `UMBRAL_GRIS` | `3` | Diferencia media máxima entre canales para considerar que la foto es gris. |
| `SAT_MIN`, `BRILLO_MIN` | `0,35`, `0,25` | Saturación y brillo mínimos (HSV) para considerar que un píxel tiene un color definido. |
| `RANGOS_TONO` | grados | Rangos de tono (H en HSV, 0–360°) de cada familia de color: rojo (330–20°), amarillo/naranja (20–70°), verde (70–170°) y azul (170–270°). |

**`analizar_pixeles(ruta)`** — abre la imagen, la convierte a RGB y devuelve:
1. **Medidas:** profundidad de color; si es gris; brillo (media de la luminancia 0,299 R + 0,587 G + 0,114 B), contraste (desviación estándar), percentiles y rango dinámico; media de cada canal; saturación; niveles de gris distintos y entropía; porcentaje de píxeles casi negros y casi blancos; **porcentaje de píxeles de cada familia de color**; y, para cada tipo de daltonismo:
    - `perdida_<tipo>`: diferencia de color media **ΔE (CIE76, espacio Lab)** entre la imagen original y la simulada. Un ΔE de 2,3 es la diferencia mínima que una persona alcanza a notar; valores altos indican que la persona percibe la imagen muy distinta a como es.
    - `cambio_correccion_<tipo>`: ΔE medio entre lo que la persona percibe de la imagen original y lo que percibe de la imagen corregida. Mide cuánto modifica la corrección la imagen **tal como la ve la persona**.
2. **El histograma de cada canal**, que se acumula por categoría.
3. **La imagen reducida a `LADO_IMAGEN_MEDIA`**, que se acumula por clase para la imagen promedio.

El cálculo de daltonismo se hace sobre una versión reducida a `LADO_DALTONISMO` px para que sea rápido. La tabla **`pixeles`** se guarda en `detalle_pixeles_por_imagen.csv`.

In [ ]:
UMBRAL_NEGRO, UMBRAL_BLANCO = 10, 245
UMBRAL_GRIS = 3
SAT_MIN, BRILLO_MIN = 0.35, 0.25
RANGOS_TONO = {"rojo": [(330, 360), (0, 20)], "amarillo_naranja": [(20, 70)],
               "verde": [(70, 170)], "azul": [(170, 270)]}

BITS_POR_CANAL = {"1": 1, "L": 8, "P": 8, "LA": 8, "RGB": 8, "RGBA": 8, "CMYK": 8,
                  "YCbCr": 8, "I;16": 16, "I": 32, "F": 32}
TIPO_COLOR = {"1": "Binaria", "L": "Escala de grises", "LA": "Grises + alfa", "P": "Paleta de colores",
              "RGB": "Color RGB", "RGBA": "Color RGB + alfa", "CMYK": "Color CMYK"}


def analizar_pixeles(ruta):
    with Image.open(ruta) as img:
        modo, bandas = img.mode, img.getbands()
        rgb = img.convert("RGB")
        hsv = np.asarray(rgb.convert("HSV"), dtype=np.float32) / 255
        miniatura = np.asarray(ImageOps.fit(rgb, (LADO_IMAGEN_MEDIA, LADO_IMAGEN_MEDIA)), dtype=np.float64)
        reducida = np.asarray(ImageOps.fit(rgb, (LADO_DALTONISMO, LADO_DALTONISMO)))
        arr = np.asarray(rgb, dtype=np.float32)

    r, g, b = arr[..., 0], arr[..., 1], arr[..., 2]
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    lum8 = np.clip(lum.round(), 0, 255).astype(np.uint8)
    hist_lum = np.bincount(lum8.ravel(), minlength=256)
    prob = hist_lum[hist_lum > 0] / lum8.size
    dif_canales = (np.abs(r - g) + np.abs(r - b) + np.abs(g - b)).mean() / 3
    p1, p5, p50, p95, p99 = np.percentile(lum, [1, 5, 50, 95, 99])
    canales = len(bandas)

    tono, sat, val = hsv[..., 0] * 360, hsv[..., 1], hsv[..., 2]
    cromatico = (sat >= SAT_MIN) & (val >= BRILLO_MIN)
    presencia = {}
    for nombre, rangos in RANGOS_TONO.items():
        en_rango = np.zeros_like(cromatico)
        for lo, hi in rangos:
            en_rango |= (tono >= lo) & (tono < hi)
        presencia[f"pct_{nombre}"] = 100 * (cromatico & en_rango).mean()

    daltonismo = {}
    for tipo in TIPOS:
        percibida = dal.simular(reducida, tipo)
        percibida_corregida = dal.simular(dal.corregir(reducida, tipo), tipo)
        daltonismo[f"perdida_{tipo}"] = float(dal.delta_e(reducida, percibida).mean())
        daltonismo[f"cambio_correccion_{tipo}"] = float(dal.delta_e(percibida, percibida_corregida).mean())

    medidas = {
        "modo": modo, "canales": canales, "bits_por_canal": BITS_POR_CANAL.get(modo, np.nan),
        "bits_por_pixel": BITS_POR_CANAL.get(modo, np.nan) * canales,
        "dif_media_canales": dif_canales, "escala_grises": dif_canales < UMBRAL_GRIS,
        "brillo": lum.mean(), "contraste": lum.std(),
        "intensidad_p05": p5, "intensidad_mediana": p50, "intensidad_p95": p95,
        "rango_dinamico": p99 - p1,
        "media_R": r.mean(), "media_G": g.mean(), "media_B": b.mean(),
        "saturacion_pct": 100 * sat.mean(),
        "niveles_distintos": int((hist_lum > 0).sum()),
        "entropia_bits": float(-(prob * np.log2(prob)).sum()),
        "pct_negro": 100 * (lum < UMBRAL_NEGRO).mean(),
        "pct_blanco": 100 * (lum > UMBRAL_BLANCO).mean(),
        **presencia, **daltonismo,
    }
    hist_rgb = np.stack([np.bincount(c.astype(np.uint8).ravel(), minlength=256) for c in (r, g, b)])
    return medidas, hist_rgb, miniatura


filas = []
categorias = sorted(datos["categoria"].unique())
hist_por_categoria = {c: np.zeros((3, 256)) for c in categorias}
suma_por_clase = {c: np.zeros((LADO_IMAGEN_MEDIA, LADO_IMAGEN_MEDIA, 3)) for c in clases}
n_por_clase = {c: 0 for c in clases}

print(f"Analizando {len(datos)} imágenes (puede tardar unos minutos si están en Drive)...")
for i, (_, fila) in enumerate(datos.iterrows(), 1):
    try:
        medidas, hist_rgb, miniatura = analizar_pixeles(fila["ruta"])
    except Exception as e:
        print(f"  No se pudo analizar {fila['archivo']}: {e}")
        continue
    filas.append({"archivo": fila["archivo"], "split": fila["split"], "clase": fila["clase"],
                  "categoria": fila["categoria"], **medidas})
    hist_por_categoria[fila["categoria"]] += hist_rgb
    suma_por_clase[fila["clase"]] += miniatura
    n_por_clase[fila["clase"]] += 1
    if i % 500 == 0:
        print(f"  {i}/{len(datos)}")

pixeles = pd.DataFrame(filas).merge(
    datos[["archivo", "split", "clase", "ancho_px", "alto_px", "lado_corto", "lado_corto_original"]],
    on=["archivo", "split", "clase"], how="left")
guardar(pixeles, "detalle_pixeles_por_imagen")
print(f"Listo: {len(pixeles)} imágenes analizadas.")

### Profundidad de Color

**Qué hace el chunk.** Resume por clase el modo de color, los canales, los bits por píxel y cuántas imágenes son en realidad grises. Se guarda en `tabla_profundidad_color.csv`.

**Salida.** *(Completar con los resultados al ejecutar.)* Se espera que todas sean color RGB de 24 bits. Si aparecen imágenes en escala de grises, en ellas el color no aporta información y el modelo solo puede usar la forma.

In [ ]:
pixeles["tipo_color"] = pixeles["modo"].map(TIPO_COLOR).fillna(pixeles["modo"])
profundidad = (pixeles.groupby(["clase", "modo", "tipo_color", "canales", "bits_por_canal", "bits_por_pixel"])
                      .agg(imagenes=("archivo", "count"),
                           escala_grises=("escala_grises", "sum"),
                           dif_media_canales=("dif_media_canales", "mean"))
                      .reset_index().set_index("clase").round(1))

print("Profundidad de color por clase")
print("  bits_por_pixel = bits por canal × número de canales")
print(f"  escala_grises = fotos guardadas en color cuyo contenido es gris (diferencia entre canales < {UMBRAL_GRIS})")
display(guardar(profundidad, "tabla_profundidad_color"))

### Pixeles y Color por Clase

**Qué hace el chunk.** Promedia por clase las medidas de intensidad, color y presencia de cada familia de color, y las guarda en `tabla_pixeles_clase.csv` y `tabla_presencia_color.csv`.

**Salida.** *(Completar con los resultados al ejecutar.)* Puntos a revisar:
- **Brillo y contraste:** GTSRB tiene muchas tomas oscuras; un brillo medio bajo o un porcentaje alto de píxeles casi negros indica imágenes difíciles.
- **Presencia de color:** las clases reglamentarias deberían tener el mayor porcentaje de rojo (borde y fondo del PARE), `velocidad_maxima` y `no_entre` también; las preventivas alemanas también tienen borde rojo, lo que confirma la limitación descrita en la preparación del dataset (en Colombia serían amarillas).

In [ ]:
metricas_px = ["brillo", "contraste", "intensidad_p05", "intensidad_mediana", "intensidad_p95",
               "rango_dinamico", "media_R", "media_G", "media_B", "saturacion_pct",
               "niveles_distintos", "entropia_bits", "pct_negro", "pct_blanco"]
metricas_color = [f"pct_{c}" for c in RANGOS_TONO]

px_clase = pixeles.groupby("clase")[metricas_px].mean().round(2)
presencia_color = pixeles.groupby(["categoria", "clase"])[metricas_color].mean().round(2)

print("Descripción de los píxeles (promedio por imagen; intensidad y canales de 0 = negro a 255 = blanco)")
display(guardar(px_clase, "tabla_pixeles_clase"))
print(f"Presencia de color: % de píxeles con saturación ≥ {SAT_MIN:.0%} y brillo ≥ {BRILLO_MIN:.0%} en cada rango de tono")
display(guardar(presencia_color, "tabla_presencia_color"))

### Pérdida de Color por Tipo de Daltonismo

**Qué hace el chunk.** Resume por clase y tipo de daltonismo:
- `perdida_<tipo>`: ΔE medio entre la imagen original y la simulada, es decir, cuánta información de color pierde la persona;
- `cambio_correccion_<tipo>`: ΔE medio entre lo que la persona percibe de la imagen original y de la corregida, es decir, cuánto le cambia la corrección.

Se guarda en `tabla_perdida_daltonismo.csv` y se grafica como mapa de calor (`perdida_daltonismo.png`).

**Salida.** *(Completar con los resultados al ejecutar.)* Se espera que la **protanopía y la deuteranopía** produzcan la mayor pérdida en las clases con más rojo (reglamentarias), porque el rojo y el verde se confunden; la tritanopía afecta sobre todo a azules y amarillos. Si `cambio_correccion` es alto donde la pérdida es alta, la corrección está actuando donde más se necesita.

In [ ]:
cols_perdida = [f"perdida_{t}" for t in TIPOS]
cols_cambio = [f"cambio_correccion_{t}" for t in TIPOS]
perdida = pixeles.groupby("clase")[cols_perdida + cols_cambio].mean().round(2)
print("ΔE medio (CIE76). Referencia: 2,3 ≈ diferencia apenas perceptible")
display(guardar(perdida, "tabla_perdida_daltonismo"))

fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * len(clases) + 1.6), sharey=True)
for ax, cols, titulo in [(axes[0], cols_perdida, "Color perdido (original vs. simulada)"),
                         (axes[1], cols_cambio, "Cambio percibido con la corrección")]:
    valores = perdida[cols].to_numpy()
    im = ax.imshow(valores, cmap="Oranges", aspect="auto")
    ax.set_xticks(range(len(TIPOS)), [dal.TIPOS[t].split(" ")[0] for t in TIPOS])
    ax.set_yticks(range(len(perdida)), perdida.index)
    for (i, j), v in np.ndenumerate(valores):
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8,
                color="white" if v > valores.max() * 0.6 else "#0b0b0b")
    ax.set_title(titulo, fontsize=11)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8, label="ΔE medio")
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "perdida_daltonismo.png", dpi=150, bbox_inches="tight")
plt.show()

### Gráficos de Descripción de Imágenes

**Qué hacen los chunks.** Cada celda guarda su imagen en `CARPETA_DESCRIPTIVOS`.

1. **Distribución de los canales de color por categoría** (`histograma_rgb.png`): un panel por canal con una línea por categoría (reglamentaria, preventiva y transitoria).
2. **Brillo, contraste y saturación por imagen** (`brillo_contraste_saturacion.png`): diagramas de caja por clase.
3. **Presencia de color por clase** (`presencia_color.png`): barras apiladas con el porcentaje de píxeles rojos, amarillo/naranja, verdes y azules.
4. **Imagen promedio por clase** (`imagen_promedio_clase.png`): como los recortes de GTSRB están centrados en la señal, el promedio sí deja ver la **forma** típica de cada clase (círculo, triángulo, octágono) y su color dominante.
5. **Simulación y corrección** (`simulacion_correccion.png`): una imagen de algunas clases tal como la percibe una persona con cada tipo de daltonismo, sin corrección y con corrección.

**Salida.** *(Completar con los resultados al ejecutar.)*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
niveles_x = np.arange(256)
techo = 1.3 * max((100 * h[:, 3:253] / h.sum(axis=1, keepdims=True)).max() for h in hist_por_categoria.values())
for i, (ax, canal) in enumerate(zip(axes, ["Rojo (R)", "Verde (G)", "Azul (B)"])):
    for c in categorias:
        densidad = 100 * hist_por_categoria[c][i] / max(hist_por_categoria[c][i].sum(), 1)
        ax.plot(niveles_x, densidad, color=COLOR_CATEGORIA.get(c, "#888"), linewidth=1.8, label=c)
    ax.set_xlim(0, 255)
    ax.set_ylim(0, techo)
    ax.set_title(f"Canal {canal}", fontsize=11)
    ax.set_xlabel("Nivel (0 = oscuro, 255 = claro)")
axes[0].set_ylabel("% de los píxeles")
axes[0].legend(frameon=False)
fig.suptitle("Distribución de los canales de color por categoría")
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "histograma_rgb.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
grupos = [(c, pixeles[pixeles["clase"] == c]) for c in clases]
fig, axes = plt.subplots(1, 3, figsize=(15, 0.42 * len(clases) + 1.5))
for ax, col, titulo, eje in [(axes[0], "brillo", "Brillo medio por imagen", "Intensidad (0-255)"),
                             (axes[1], "contraste", "Contraste por imagen (desv. estándar)", "Intensidad (0-255)"),
                             (axes[2], "saturacion_pct", "Saturación media por imagen", "Saturación (%)")]:
    bp = ax.boxplot([g[col] for _, g in grupos], vert=False, widths=0.5, patch_artist=True,
                    medianprops={"color": "#0b0b0b", "linewidth": 1.5},
                    flierprops={"marker": "o", "markersize": 2, "alpha": 0.4})
    for caja, (c, _) in zip(bp["boxes"], grupos):
        caja.set_facecolor(COLOR_CLASE.get(c, "#888"))
        caja.set_alpha(0.35)
        caja.set_edgecolor(COLOR_CLASE.get(c, "#888"))
    ax.set_yticks(range(1, len(grupos) + 1), [c for c, _ in grupos])
    ax.set_xlabel(eje)
    ax.set_title(titulo, fontsize=11)
    ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "brillo_contraste_saturacion.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
COLOR_FAMILIA = {"pct_rojo": "#d62828", "pct_amarillo_naranja": "#f2a900",
                 "pct_verde": "#2a9d4b", "pct_azul": "#1d4ed8"}
tabla = pixeles.groupby("clase")[metricas_color].mean().loc[clases[::-1]]
fig, ax = plt.subplots(figsize=(9, 0.42 * len(clases) + 1.2))
izquierda = np.zeros(len(tabla))
for col in metricas_color:
    ax.barh(tabla.index, tabla[col], left=izquierda, color=COLOR_FAMILIA[col],
            label=col.replace("pct_", "").replace("_", "/"))
    izquierda += tabla[col].to_numpy()
ax.set_xlabel("% de píxeles de la imagen")
ax.set_title("Presencia de color por clase")
ax.legend(frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.06))
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "presencia_color.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
columnas = 5
filas_fig = int(np.ceil(len(clases) / columnas))
fig, axes = plt.subplots(filas_fig, columnas, figsize=(2.6 * columnas, 2.9 * filas_fig), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, c in zip(axes.ravel(), clases):
    media = (suma_por_clase[c] / max(n_por_clase[c], 1)).astype(np.uint8)
    ax.imshow(media)
    ax.set_title(f"{c}\n(n={n_por_clase[c]})", fontsize=9)
fig.suptitle(f"Imagen promedio por clase ({LADO_IMAGEN_MEDIA}×{LADO_IMAGEN_MEDIA} px)")
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "imagen_promedio_clase.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
CLASES_EJEMPLO = [c for c in ["pare", "ceda_el_paso", "obras", "semaforo_adelante"] if c in clases] or clases[:4]
columnas = ["Original"] + [f"{dal.TIPOS[t].split(' ')[0]}\n{etq}" for t in TIPOS for etq in ("sin corrección", "con corrección")]
fig, axes = plt.subplots(len(CLASES_EJEMPLO), len(columnas),
                         figsize=(1.75 * len(columnas), 1.95 * len(CLASES_EJEMPLO)), squeeze=False)
for fila, c in enumerate(CLASES_EJEMPLO):
    muestra = datos[(datos["clase"] == c) & (datos["split"] == "train")].sample(1, random_state=3).iloc[0]
    with Image.open(muestra["ruta"]) as img:
        arr = np.asarray(ImageOps.fit(img.convert("RGB"), (160, 160)))
    vistas = [arr]
    for t in TIPOS:
        vistas += [dal.simular(arr, t), dal.simular(dal.corregir(arr, t), t)]
    for col, vista in enumerate(vistas):
        ax = axes[fila, col]
        ax.imshow(vista)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        if fila == 0:
            ax.set_title(columnas[col], fontsize=8)
        if col == 0:
            ax.set_ylabel(c, fontsize=8)
fig.suptitle("Cómo percibe la señal una persona con daltonismo, sin y con la corrección de la aplicación")
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "simulacion_correccion.png", dpi=150, bbox_inches="tight")
plt.show()

### Resumen

**Qué hace el chunk.** Imprime un resumen por clase con la profundidad de color, las imágenes en gris, la resolución original, el brillo, contraste, saturación, el color dominante y la pérdida de color por tipo de daltonismo.

In [ ]:
print("RESUMEN DE PÍXELES, RESOLUCIÓN Y COLOR")
for c in clases:
    sub = pixeles[pixeles["clase"] == c]
    dominante = sub[metricas_color].mean().idxmax().replace("pct_", "")
    perd = ", ".join(f"{dal.TIPOS[t].split(' ')[0]} {sub[f'perdida_{t}'].mean():.1f}" for t in TIPOS)
    print(f"\n  {c} · {CATEGORIA.get(c, ('-',))[0]} · {len(sub)} imágenes")
    print(f"    Profundidad de color:  {', '.join(f'{m}: {n}' for m, n in sub['modo'].value_counts().items())}"
          f" · en gris: {int(sub['escala_grises'].sum())}")
    print(f"    Lado corto original:   mediana {sub['lado_corto_original'].median():.0f} px "
          f"(mín. {sub['lado_corto_original'].min():.0f}, máx. {sub['lado_corto_original'].max():.0f})")
    print(f"    Intensidad y color:    brillo {sub['brillo'].mean():.1f}, contraste {sub['contraste'].mean():.1f}, "
          f"saturación {sub['saturacion_pct'].mean():.1f} %, color dominante: {dominante}")
    print(f"    ΔE perdido:            {perd}")
print(f"\nTablas y gráficos guardados en: {CARPETA_DESCRIPTIVOS}")

## Descriptivo Sobre Memoria

**Qué hace el chunk.** Resume la memoria que ocupan las imágenes por clase y conjunto, y la guarda en `tabla_memoria.csv`:

- **disco**: lo que ocupan los JPG (comprimidos): total, media, mediana, mínimo y máximo;
- **ram**: lo que ocupan al decodificarlas (ancho × alto × 3 bytes);
- **supera_limite_prediccion**: cuántas pesan más que `MAX_MB_PREDICCION`.

**Salida.** *(Completar con los resultados al ejecutar.)*

In [ ]:
def resumen_memoria(df):
    return pd.Series({
        "imagenes": len(df),
        "disco_total_mb": df["disco_mb"].sum(), "disco_media_mb": df["disco_mb"].mean(),
        "disco_mediana_mb": df["disco_mb"].median(), "disco_min_mb": df["disco_mb"].min(),
        "disco_max_mb": df["disco_mb"].max(),
        "ram_total_mb": df["ram_mb"].sum(), "ram_media_mb": df["ram_mb"].mean(),
        "supera_limite_prediccion": int((df["disco_mb"] > MAX_MB_PREDICCION).sum()),
    })


memoria = pd.DataFrame({c: resumen_memoria(g) for c, g in datos.groupby("clase")}).T
for split, g in datos.groupby("split"):
    memoria.loc[f"total {split}"] = resumen_memoria(g)
memoria.loc["total"] = resumen_memoria(datos)
memoria = memoria.round(3)

print("Memoria en MB")
print("  disco: lo que ocupan los JPG · ram: lo que ocupan al decodificarlos")
print(f"  supera_limite_prediccion: imágenes de más de {MAX_MB_PREDICCION} MB")
display(guardar(memoria, "tabla_memoria"))

## Resumen Gráfico

**Qué hacen los chunks.** Cada celda guarda su imagen en `CARPETA_DESCRIPTIVOS`:

1. **Distribución del peso de las imágenes por categoría** (`distribucion_peso.png`).
2. **Dimensiones originales** (`dimensiones.png`): dispersión de ancho contra alto de los recortes de GTSRB, por categoría, con la diagonal (imagen cuadrada) y la línea de 256 px. Muestra cuánto se tuvieron que ampliar.
3. **Ejemplos por clase** (`ejemplos_por_clase.png`): cinco imágenes al azar (`random_state=0`) de cada clase, para revisar visualmente que cada carpeta contiene la señal correcta.

**Salida.** *(Completar con los resultados al ejecutar.)*

In [ ]:
fig, axes = plt.subplots(len(categorias), 1, figsize=(8, 1.9 * len(categorias) + 0.8), sharex=True, squeeze=False)
bins = np.linspace(0, datos["disco_mb"].max() * 1.02, 30)
for ax, c in zip(axes[:, 0], categorias):
    sub = datos.loc[datos["categoria"] == c, "disco_mb"]
    ax.hist(sub, bins=bins, color=COLOR_CATEGORIA.get(c, "#888"), edgecolor="white", linewidth=0.8)
    ax.axvline(sub.median(), color="#52514e", linewidth=1, linestyle="--")
    ax.text(0.99, 0.9, f"{c} · n={len(sub)} · mediana {sub.median():.3f} MB",
            transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#52514e")
    ax.set_ylabel("Imágenes")
    ax.grid(axis="x", visible=False)
axes[-1, 0].set_xlabel("Peso en disco (MB por imagen)")
fig.suptitle("Distribución del peso de las imágenes")
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "distribucion_peso.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
lim = [0, max(datos["ancho_original"].max(), datos["alto_original"].max(), LADO_MINIMO_CV) * 1.05]
for c in categorias:
    sub = datos[datos["categoria"] == c]
    ax.scatter(sub["ancho_original"], sub["alto_original"], s=14, alpha=0.4,
               color=COLOR_CATEGORIA.get(c, "#888"), edgecolor="none", label=f"{c} (n={len(sub)})")
ax.plot(lim, lim, color="#9a9a9a", linewidth=1, linestyle="--", label="imagen cuadrada")
ax.axhline(LADO_MINIMO_CV, color="#c9c9c9", linewidth=1)
ax.axvline(LADO_MINIMO_CV, color="#c9c9c9", linewidth=1)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("Ancho original (px)")
ax.set_ylabel("Alto original (px)")
ax.set_title(f"Dimensiones originales de los recortes (líneas grises: {LADO_MINIMO_CV} px)")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "dimensiones.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
N_EJEMPLOS = 5
fig, axes = plt.subplots(len(clases), N_EJEMPLOS, figsize=(1.9 * N_EJEMPLOS, 1.9 * len(clases)), squeeze=False)
for fila, c in enumerate(clases):
    sub = datos[(datos["clase"] == c) & (datos["split"] == "train")]
    ejemplos = sub.sample(n=min(N_EJEMPLOS, len(sub)), random_state=0)
    for col in range(N_EJEMPLOS):
        ax = axes[fila, col]
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        if col >= len(ejemplos):
            ax.axis("off")
            continue
        r = ejemplos.iloc[col]
        with Image.open(r["ruta"]) as img:
            img = img.convert("RGB")
            img.thumbnail((160, 160))
            ax.imshow(img)
        ax.set_title(f"{int(r['ancho_original'])}×{int(r['alto_original'])}", fontsize=7)
        if col == 0:
            ax.set_ylabel(c, fontsize=8)
fig.suptitle("Ejemplos de imágenes por clase (tamaño original)")
fig.tight_layout()
fig.savefig(CARPETA_DESCRIPTIVOS / "ejemplos_por_clase.png", dpi=150, bbox_inches="tight")
plt.show()

## Resumen General

**Qué hace el chunk.** Imprime un resumen de una línea por indicador.

In [ ]:
print("RESUMEN")
print(f"  Imágenes descritas:          {len(datos)} (entrenamiento {int((datos['split'] == 'train').sum())}, "
      f"prueba {int((datos['split'] == 'test').sum())})")
print(f"  Clases:                      {len(clases)} ({', '.join(clases)})")
print(f"  Categorías:                  {', '.join(f'{c}: {n}' for c, n in datos['categoria'].value_counts().items())}")
print(f"  Lado corto original:         mediana {datos['lado_corto_original'].median():.0f} px")
print(f"  Imágenes ampliadas:          {int(datos['ampliada'].sum())} ({100 * datos['ampliada'].mean():.1f} %)")
print(f"  Peso total en disco:         {datos['disco_mb'].sum():,.2f} MB")
print(f"  Memoria al decodificarlas:   {datos['ram_mb'].sum():,.1f} MB")
print(f"  ΔE medio perdido:            " + ", ".join(f"{dal.TIPOS[t]} {pixeles[f'perdida_{t}'].mean():.1f}" for t in TIPOS))
print(f"\nTablas y gráficos guardados en: {CARPETA_DESCRIPTIVOS}")

# Azure Custom Vision

**Contexto en el desarrollo del modelo.** El entrenamiento se hace **fuera del notebook**, en [Azure AI Custom Vision](https://www.customvision.ai/). El servicio usa **aprendizaje por transferencia**: parte de una red preentrenada con millones de imágenes y ajusta la parte final para distinguir las 10 señales. El modelo entrenado se **publica como un servicio web (API REST)** con una URL y una clave, que es lo que consumen tanto este notebook como la aplicación web.

Solo se cargan las imágenes de `train/`. Las de `test/` se reservan para la evaluación independiente de la sección **Evaluación del Modelo**, porque las métricas que muestra Custom Vision se calculan sobre una partición interna de las mismas imágenes de entrenamiento.

| Elemento | Valor |
| --- | --- |
| Recurso de Azure | *(nombre, nivel de precios, región y grupo de recursos)* |
| Proyecto | *(nombre del proyecto)* |
| Tipo de proyecto | Clasificación **multiclase** (una etiqueta por imagen) |
| Dominio | *(General [A2], o General (compact) si se quiere exportar el modelo)* |
| Etiquetas | 10, con 150 imágenes cada una (carpetas de `train/`) |
| Entrenamiento | *(Iteración 1: Quick Training; Iteración 2: …)* |
| Iteración publicada | *(nombre de la iteración publicada)* |

## Desarrollo de Modelo

**Cómo llegan las capturas al repositorio.** Los pantallazos tomados durante el entrenamiento se importan desde la terminal:

```bash
python vv.py capturas --origen <carpeta_pantallazos> --plantilla   # crea descripciones.csv para completar
python vv.py capturas --origen <carpeta_pantallazos>               # copia, renombra (Paso_01.png …) y crea capturas.json
```

**Qué hace el chunk.** Lee `assets/entrenamiento/capturas.json` y muestra en orden cada captura con su título y descripción. Es el mismo índice que muestra la pestaña *Entrenamiento en ACV* de la aplicación.

**Lo que deben mostrar las capturas** (orden sugerido en la plantilla):
1. Implementación del recurso en Azure.
2. Creación del recurso de Custom Vision.
3. Creación del proyecto (tipo, clasificación multiclase y dominio).
4. Carga de las imágenes de una etiqueta (se repite con las 10 carpetas de `train/`).
5. Imágenes cargadas y etiquetadas: 10 etiquetas con su número de imágenes.
6. Selección del tipo de entrenamiento.
7. Resultados de la iteración 1: precision, recall y AP por etiqueta.
8. Resultados de la iteración 2 (mejora: más tiempo de entrenamiento o ajustes del dataset).
9. Publicación de la iteración.
10. Datos del endpoint (*Prediction URL*).
11. Prueba rápida (*Quick Test*) con una imagen nueva.
12. Prueba rápida con una foto tomada en Bogotá.

In [ ]:
def mostrar_capturas(carpeta):
    indice = cap.leer_indice(carpeta)
    if not indice:
        display(Markdown(f"No hay capturas en `{carpeta}`. Impórtelas con `python vv.py capturas --origen <carpeta>`."))
        return
    for c in indice:
        display(Markdown(f"### {c['titulo']}" + (f"\n\n{c['descripcion']}" if c.get("descripcion") else "")))
        display(IPImage(filename=str(carpeta / c["archivo"]), width=900))


mostrar_capturas(CARPETA_SNIP)

# Uso del Servicio

**Contexto.** Al publicar la iteración, Custom Vision expone el modelo como un servicio web (*Prediction API*) con dos endpoints:

| Endpoint | Cuándo usarlo | Cómo se envía la imagen |
| --- | --- | --- |
| `.../classify/iterations/<iteración>/url` | La imagen está publicada en internet. | JSON `{"Url": "https://..."}` con `Content-Type: application/json`. |
| `.../classify/iterations/<iteración>/image` | La imagen está en el equipo, en Drive o en memoria. | Los bytes del archivo con `Content-Type: application/octet-stream`. |

Ambos requieren el encabezado **`Prediction-Key`**. La respuesta es un JSON con la lista `predictions`: una entrada por etiqueta con su nombre (`tagName`) y su probabilidad (`probability`).

## Configuración del Servicio

**Qué hace el chunk.** Define la dirección del modelo publicado y la clave de predicción.

| Parámetro | Qué es |
| --- | --- |
| `ENDPOINT_BASE` | URL de la iteración publicada, sin el sufijo `/url` o `/image`. |
| `ENDPOINT_URL`, `ENDPOINT_IMAGEN` | Endpoints para clasificar por URL o por archivo. |
| `PREDICTION_KEY` | Clave de predicción del recurso. |

**Sobre la clave.** La celda busca `CV_PREDICTION_ENDPOINT` y `CV_PREDICTION_KEY` en los **Secretos de Colab** (ícono de la llave), luego en `.streamlit/secrets.toml` del repositorio (el mismo archivo que usa la aplicación) y en variables de entorno. Si no las encuentra, usa los valores escritos en la celda, que deben reemplazarse. Para compartir el notebook, se recomienda dejar la clave solo en los Secretos o regenerarla en Azure después de la entrega.

In [ ]:
def _secreto(nombre):
    try:
        from google.colab import userdata          # Secretos de Colab
        valor = userdata.get(nombre)
        if valor:
            return valor, "Secretos de Colab"
    except Exception:
        pass
    valor = config.leer_secreto(nombre)              # .streamlit/secrets.toml del repo o variable de entorno
    if valor:
        return valor, "secrets.toml / variable de entorno"
    return None, "valor escrito en la celda"


ENDPOINT_BASE, origen_endpoint = _secreto("CV_PREDICTION_ENDPOINT")
PREDICTION_KEY, origen_clave = _secreto("CV_PREDICTION_KEY")

# Reemplazar si no se usan Secretos de Colab ni variables de entorno
ENDPOINT_BASE = ENDPOINT_BASE or ("https://<region>.api.cognitive.microsoft.com/customvision/v3.0/Prediction/"
                                  "<project-id>/classify/iterations/<iteracion>")
PREDICTION_KEY = PREDICTION_KEY or "<prediction-key>"

ENDPOINT_BASE = ENDPOINT_BASE.rstrip("/").removesuffix("/image").removesuffix("/url")
ENDPOINT_URL = f"{ENDPOINT_BASE}/url"
ENDPOINT_IMAGEN = f"{ENDPOINT_BASE}/image"

print("Endpoint por URL:   ", ENDPOINT_URL, f"(tomado de: {origen_endpoint})")
print("Endpoint por imagen:", ENDPOINT_IMAGEN)
print(f"Prediction-Key:      {PREDICTION_KEY[:4]}…{PREDICTION_KEY[-4:]} (tomada de: {origen_clave})")

## Funciones de Predicción

**Qué hace el chunk.** Define las funciones que hablan con el servicio. No ejecuta ninguna predicción.

- **`_llamar(...)`** — hace la petición `POST` con el encabezado `Prediction-Key`. Si el servicio responde **429** (demasiadas solicitudes por segundo), espera y reintenta con tiempos crecientes. Ante otro error lanza un `RuntimeError` con el mensaje de Azure (401 clave inválida, 404 iteración no publicada, 400 imagen inválida).
- **`predecir_url(url)`** — clasifica una imagen publicada en internet.
- **`preparar_bytes(origen)`** — acepta una ruta, bytes, una imagen PIL o una matriz `numpy` (por ejemplo, una imagen con daltonismo simulado) y devuelve los bytes JPEG listos para enviar, reduciéndolos si superan `MAX_MB_PREDICCION`.
- **`predecir_imagen(origen)`** — envía esos bytes al endpoint de imágenes.
- **`tabla_predicciones(respuesta)`** y **`mostrar_prediccion(...)`** — ordenan las probabilidades y dibujan la imagen junto a un gráfico de barras, con la categoría y el color de la señal ganadora.
- **`predecir_conjunto(registros, condicion, transformar=None)`** — clasifica un conjunto de imágenes, opcionalmente transformadas (simulación o corrección), y devuelve una fila por imagen con la clase real y la probabilidad de cada clase. Guarda las respuestas en `CARPETA_RESULTADOS/predicciones_<condicion>.csv` a medida que avanza: si se vuelve a ejecutar, solo consulta las imágenes que faltan. Esto evita repetir llamadas (que tienen costo) y permite retomar si se interrumpe.

In [ ]:
PAUSA_ENTRE_LLAMADAS = 0.15    # s; el nivel S0 admite ~10 solicitudes por segundo y el F0 ~2


def _llamar(endpoint, reintentos=6, **kwargs):
    headers = {"Prediction-Key": PREDICTION_KEY, **kwargs.pop("headers", {})}
    for intento in range(reintentos):
        r = requests.post(endpoint, headers=headers, timeout=60, **kwargs)
        if r.status_code != 429:
            break
        time.sleep(2 ** intento)
    if not r.ok:
        try:
            detalle = r.json().get("message", r.text)
        except ValueError:
            detalle = r.text
        raise RuntimeError(f"El servicio respondió {r.status_code}: {detalle}")
    return r.json()


def predecir_url(url_imagen):
    """Clasifica una imagen publicada en internet."""
    return _llamar(ENDPOINT_URL, json={"Url": url_imagen.strip()})


def preparar_bytes(origen):
    """Ruta, bytes, imagen PIL o matriz numpy -> bytes JPEG dentro del límite del servicio."""
    if isinstance(origen, (str, Path)):
        contenido = Path(origen).read_bytes()
    elif isinstance(origen, (bytes, bytearray)):
        contenido = bytes(origen)
    else:
        img = Image.fromarray(origen) if isinstance(origen, np.ndarray) else origen
        buffer = io.BytesIO()
        img.convert("RGB").save(buffer, format="JPEG", quality=95)
        contenido = buffer.getvalue()
    if len(contenido) <= MAX_MB_PREDICCION * 1e6:
        return contenido
    img = Image.open(io.BytesIO(contenido)).convert("RGB")
    img.thumbnail((1600, 1600))
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=90)
    return buffer.getvalue()


def predecir_imagen(origen):
    """Clasifica un archivo de imagen (ruta, bytes, PIL o numpy)."""
    return _llamar(ENDPOINT_IMAGEN, data=preparar_bytes(origen),
                   headers={"Content-Type": "application/octet-stream"})


def tabla_predicciones(respuesta):
    tabla = pd.DataFrame(respuesta["predictions"])[["tagName", "probability"]]
    tabla = tabla.rename(columns={"tagName": "clase", "probability": "probabilidad"})
    tabla["probabilidad_%"] = (100 * tabla["probabilidad"]).round(2)
    tabla["categoria"] = tabla["clase"].map(lambda c: CATEGORIA.get(c, ("-", "-"))[0])
    return tabla.sort_values("probabilidad", ascending=False).reset_index(drop=True)


def mostrar_prediccion(respuesta, imagen=None, titulo="", top=5):
    tabla = tabla_predicciones(respuesta)
    ganadora = tabla.iloc[0]
    categoria, color = CATEGORIA.get(ganadora["clase"], ("-", "-"))
    paneles = 2 if imagen is not None else 1
    fig, axes = plt.subplots(1, paneles, figsize=(5.5 * paneles, 3.6), squeeze=False)
    if imagen is not None:
        axes[0, 0].imshow(imagen)
        axes[0, 0].axis("off")
        axes[0, 0].set_title(titulo, fontsize=9)
    ax = axes[0, -1]
    orden = tabla.head(top).iloc[::-1]
    colores = [COLOR_CLASE.get(c, "#9a9a9a") if c == ganadora["clase"] else "#d0d0d0" for c in orden["clase"]]
    barras = ax.barh(orden["clase"], orden["probabilidad_%"], color=colores)
    ax.bar_label(barras, labels=[f"{v:.1f} %" for v in orden["probabilidad_%"]], padding=3, fontsize=9)
    ax.set_xlim(0, 115)
    ax.set_xlabel("Probabilidad (%)")
    ax.set_title(f"Predicción: {ganadora['clase']} ({ganadora['probabilidad_%']:.1f} %)\n"
                 f"{categoria} · color {color}", fontsize=10)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    plt.show()
    display(tabla.head(top))


def predecir_conjunto(registros, condicion, transformar=None, cada=50):
    """Clasifica las imágenes de 'registros' (columnas ruta, split, clase, archivo) con caché en CSV."""
    ruta_cache = CARPETA_RESULTADOS / f"predicciones_{condicion}.csv"
    registros = registros.assign(id=registros["split"] + "/" + registros["clase"] + "/" + registros["archivo"])
    previas = pd.read_csv(ruta_cache) if ruta_cache.exists() else pd.DataFrame(columns=["id"])
    pendientes = registros[~registros["id"].isin(set(previas["id"]))]
    if len(pendientes):
        print(f"[{condicion}] {len(pendientes)} imágenes por consultar ({len(registros) - len(pendientes)} en caché)")
    nuevas = []
    for i, (_, fila) in enumerate(pendientes.iterrows(), 1):
        with Image.open(fila["ruta"]) as img:
            arr = np.asarray(img.convert("RGB"))
        if transformar is not None:
            arr = transformar(arr)
        try:
            respuesta = predecir_imagen(arr)
        except Exception as e:
            print(f"  {fila['archivo']}: {e}")
            if "401" in str(e) or "404" in str(e):
                break                      # clave o iteración incorrecta: no tiene sentido seguir
            continue
        nuevas.append({"id": fila["id"], "archivo": fila["archivo"], "clase_real": fila["clase"],
                       "condicion": condicion,
                       **{p["tagName"]: p["probability"] for p in respuesta["predictions"]}})
        if i % cada == 0 or i == len(pendientes):
            pd.concat([previas, pd.DataFrame(nuevas)], ignore_index=True).to_csv(ruta_cache, index=False)
            print(f"  {i}/{len(pendientes)}")
        time.sleep(PAUSA_ENTRE_LLAMADAS)
    todas = pd.concat([previas, pd.DataFrame(nuevas)], ignore_index=True)
    todas = todas[todas["id"].isin(set(registros["id"]))].reset_index(drop=True)
    for c in clases:                       # una columna de probabilidad por clase del dataset
        if c not in todas:
            todas[c] = 0.0
    todas[clases] = todas[clases].fillna(0.0)
    return todas

## Verificación del Servicio

**Qué hace el chunk.** Comprueba que el modelo publicado responde enviando una imagen de entrenamiento de algunas clases (al azar, `random_state=1`).

**Importante.** Estas imágenes se usaron para entrenar, así que la celda **no es una evaluación**: solo confirma que la conexión, la clave y la iteración publicada funcionan.

In [ ]:
verificacion = datos[datos["split"] == "train"].groupby("clase").sample(n=1, random_state=1).head(3)

for _, fila in verificacion.iterrows():
    try:
        respuesta = predecir_imagen(fila["ruta"])
    except Exception as e:
        print(f"No se pudo consultar el servicio con {fila['archivo']}: {e}")
        break
    with Image.open(fila["ruta"]) as img:
        mostrar_prediccion(respuesta, img.convert("RGB"), titulo=f"{fila['clase']}/{fila['archivo']}")

# Evaluación del Modelo

**Contexto.** Las métricas que reporta Custom Vision se calculan sobre una partición interna de las imágenes cargadas. Para medir cómo se comporta el modelo con imágenes que **nunca vio**, se evalúa con el conjunto `test/`: 50 imágenes por clase tomadas de la partición oficial de prueba de GTSRB, que no comparte señales físicas con el entrenamiento.

Se calculan las métricas que pide la actividad:

| Métrica | Qué mide |
| --- | --- |
| **Precision** | De las imágenes que el modelo asignó a una clase, qué porcentaje era realmente de esa clase. |
| **Recall** | De las imágenes de una clase, qué porcentaje reconoció el modelo. |
| **F1-score** | Media armónica de precision y recall. |
| **Curva ROC** | Tasa de verdaderos positivos contra tasa de falsos positivos al variar el umbral de probabilidad, para cada clase contra el resto (*one-vs-rest*). |
| **AUC** | Área bajo la curva ROC: 1 = separación perfecta, 0,5 = azar. |

**Relación con la terminal.** Esta sección evalúa la iteración **publicada** a través del endpoint. El comando `python vv.py evaluar` hace la misma evaluación con el modelo **exportado en ONNX** (sin costo por predicción) y escribe `resultados/metricas.json`, que es lo que muestra la aplicación. Ambas deberían dar resultados prácticamente iguales; la celda siguiente muestra los de la terminal si ya existen.

In [ ]:
metricas_cli = config.leer_json(config.METRICAS)
if metricas_cli:
    g = metricas_cli["global"]
    print(f"Evaluación desde la terminal ({metricas_cli['modelo']['descripcion']}, {metricas_cli['evaluado']}):")
    print(f"  accuracy {g['accuracy']:.3f} · precision macro {g['precision_macro']:.3f} · recall macro {g['recall_macro']:.3f}"
          f" · F1 macro {g['f1_macro']:.3f} · AUC macro {g['auc_macro']:.3f}")
    print(f"  Umbral recomendado para la app: {metricas_cli['umbral_recomendado']:.2f}")
else:
    print("Aún no se ha ejecutado `python vv.py evaluar`.")

## Predicción del Conjunto de Prueba

**Qué hace el chunk.** Envía al servicio todas las imágenes de `test/` con `predecir_conjunto` (condición `original`) y arma:
- `y_real`: la clase verdadera;
- `P`: la matriz de probabilidades (una columna por clase);
- `y_pred`: la clase con mayor probabilidad.

La primera ejecución hace una llamada por imagen (unas 500); las siguientes leen la caché.

In [ ]:
prueba = datos[datos["split"] == "test"].reset_index(drop=True)
pred_test = predecir_conjunto(prueba, "original")


def separar(pred):
    y_real = pred["clase_real"].to_numpy()
    P = pred[clases].to_numpy(dtype=float)
    y_pred = np.array(clases)[P.argmax(axis=1)]
    return y_real, y_pred, P


y_real, y_pred, P = separar(pred_test)
print(f"Imágenes de prueba clasificadas: {len(pred_test)} de {len(prueba)}")
print(f"Accuracy: {accuracy_score(y_real, y_pred):.3f}")

## Reporte de Clasificación

**Qué hace el chunk.** Calcula precision, recall, F1-score y soporte por clase, y sus promedios *macro* (todas las clases pesan igual) y *weighted* (ponderado por el número de imágenes). Se guarda en `reporte_clasificacion.csv`.

**Salida.** *(Completar con los resultados al ejecutar.)* Identificar las clases con menor recall (las que el modelo más deja de reconocer) y las de menor precision (a las que más asigna imágenes de otras clases).

In [ ]:
reporte = pd.DataFrame(classification_report(y_real, y_pred, labels=clases, output_dict=True,
                                             zero_division=0)).T
reporte.loc["accuracy", ["precision", "recall", "f1-score"]] = accuracy_score(y_real, y_pred)
reporte.loc["accuracy", "support"] = len(y_real)
reporte["categoria"] = [CATEGORIA.get(i, ("", ""))[0] for i in reporte.index]
reporte = reporte.round(3)
display(guardar(reporte, "reporte_clasificacion", CARPETA_RESULTADOS))

fig, ax = plt.subplots(figsize=(9, 0.42 * len(clases) + 1.3))
sub = reporte.loc[clases[::-1]]
alto = 0.27
for k, (metrica, color) in enumerate([("precision", "#2a78d6"), ("recall", "#eb6834"), ("f1-score", "#1baf7a")]):
    ax.barh(np.arange(len(sub)) + (k - 1) * alto, sub[metrica], height=alto, color=color, label=metrica)
ax.set_yticks(np.arange(len(sub)), sub.index)
ax.set_xlim(0, 1.05)
ax.set_xlabel("Valor")
ax.set_title("Precision, recall y F1-score por clase (conjunto de prueba)")
ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.05))
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(CARPETA_RESULTADOS / "metricas_por_clase.png", dpi=150, bbox_inches="tight")
plt.show()

## Matriz de Confusión

**Qué hace el chunk.** Cuenta cuántas imágenes de cada clase real (filas) se asignaron a cada clase predicha (columnas). Se muestra en números absolutos y normalizada por fila (porcentaje de cada clase real). Se guarda en `matriz_confusion.csv` y `matriz_confusion.png`.

**Salida.** *(Completar con los resultados al ejecutar.)* Prestar atención a las confusiones **entre categorías** (por ejemplo, una señal reglamentaria confundida con una preventiva), que son las más graves para el usuario.

In [ ]:
mc = confusion_matrix(y_real, y_pred, labels=clases)
mc_norm = mc / np.maximum(mc.sum(axis=1, keepdims=True), 1)
guardar(pd.DataFrame(mc, index=clases, columns=clases), "matriz_confusion", CARPETA_RESULTADOS)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={"wspace": 0.08})
for ax, matriz, titulo, fmt in [(axes[0], mc, "Conteo", "{:d}"), (axes[1], mc_norm, "Normalizada por fila", "{:.0%}")]:
    im = ax.imshow(mc_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(clases)), clases, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(clases)), clases if ax is axes[0] else [""] * len(clases), fontsize=8)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real" if ax is axes[0] else "")
    ax.set_title(titulo)
    ax.grid(False)
    for (i, j), v in np.ndenumerate(matriz):
        if mc[i, j] > 0:
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=7,
                    color="white" if mc_norm[i, j] > 0.6 else "#0b0b0b")
fig.colorbar(im, ax=axes, shrink=0.8, label="Proporción de la clase real")
fig.savefig(CARPETA_RESULTADOS / "matriz_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

## Curvas ROC y AUC

**Qué hace el chunk.** Para cada clase, trata el problema como binario (esa clase contra el resto) y calcula la curva ROC a partir de la probabilidad que devuelve el servicio. Calcula también la curva **micro** (todas las decisiones juntas) y el AUC **macro** (promedio de las clases). Se guarda en `auc_por_clase.csv` y `curvas_roc.png`.

**Salida.** *(Completar con los resultados al ejecutar.)* Un AUC alto con un recall bajo en alguna clase indica que el modelo sí ordena bien las probabilidades, pero que otra clase le gana por poco: en ese caso, más imágenes de esas dos clases suelen ayudar.

In [ ]:
Y = label_binarize(y_real, classes=clases)
auc_clase, curvas = {}, {}
for i, c in enumerate(clases):
    if Y[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(Y[:, i], P[:, i])
    curvas[c] = (fpr, tpr)
    auc_clase[c] = auc(fpr, tpr)
fpr_micro, tpr_micro, _ = roc_curve(Y.ravel(), P.ravel())
auc_micro = auc(fpr_micro, tpr_micro)
auc_macro = float(np.mean(list(auc_clase.values())))

tabla_auc = pd.DataFrame({"AUC": auc_clase}).rename_axis("clase")
tabla_auc["categoria"] = [CATEGORIA.get(c, ("-",))[0] for c in tabla_auc.index]
tabla_auc.loc["micro"] = [auc_micro, ""]
tabla_auc.loc["macro"] = [auc_macro, ""]
tabla_auc["AUC"] = tabla_auc["AUC"].astype(float).round(4)
display(guardar(tabla_auc, "auc_por_clase", CARPETA_RESULTADOS))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for ax, zoom in [(axes[0], False), (axes[1], True)]:
    for c, (fpr, tpr) in curvas.items():
        ax.plot(fpr, tpr, color=COLOR_CLASE.get(c, "#888"), linewidth=1.4, label=f"{c} (AUC {auc_clase[c]:.3f})")
    ax.plot(fpr_micro, tpr_micro, color="#0b0b0b", linewidth=2, linestyle=":", label=f"micro (AUC {auc_micro:.3f})")
    ax.plot([0, 1], [0, 1], color="#bdbdbd", linestyle="--", linewidth=1)
    ax.set_xlabel("Tasa de falsos positivos")
    ax.set_ylabel("Tasa de verdaderos positivos")
    if zoom:
        ax.set_xlim(0, 0.2)
        ax.set_ylim(0.8, 1.005)
        ax.set_title("Detalle de la esquina superior izquierda")
    else:
        ax.set_title(f"Curvas ROC uno-contra-resto · AUC macro {auc_macro:.3f}")
axes[0].legend(frameon=False, fontsize=7, loc="lower right")
fig.tight_layout()
fig.savefig(CARPETA_RESULTADOS / "curvas_roc.png", dpi=150, bbox_inches="tight")
plt.show()

## Desempeño por Categoría y Errores

**Qué hace el chunk.**
- `tabla_categoria`: accuracy por categoría y **error de categoría**, es decir, el porcentaje de imágenes cuya categoría predicha (reglamentaria, preventiva o transitoria) no coincide con la real. Para una persona con daltonismo este es el error que más importa, porque la categoría es justo lo que no puede deducir del color.
- Galería de los errores con mayor confianza: las imágenes mal clasificadas en las que el modelo estaba más seguro. Sirven para entender las causas (poca luz, desenfoque, señales parecidas).

**Salida.** *(Completar con los resultados al ejecutar.)*

In [ ]:
def categoria_de(etiquetas):
    return np.array([CATEGORIA.get(e, ("-",))[0] for e in etiquetas])


resultado = pred_test.assign(prediccion=y_pred, confianza=P.max(axis=1),
                             categoria_real=categoria_de(y_real), categoria_pred=categoria_de(y_pred))
resultado["acierto"] = resultado["clase_real"] == resultado["prediccion"]
resultado["acierto_categoria"] = resultado["categoria_real"] == resultado["categoria_pred"]

tabla_categoria = resultado.groupby("categoria_real").agg(
    imagenes=("id", "count"),
    accuracy=("acierto", "mean"),
    error_categoria_pct=("acierto_categoria", lambda s: 100 * (1 - s.mean())),
).round(3)
display(guardar(tabla_categoria, "desempeno_por_categoria", CARPETA_RESULTADOS))

errores = resultado[~resultado["acierto"]].sort_values("confianza", ascending=False).head(12)
print(f"Errores: {int((~resultado['acierto']).sum())} de {len(resultado)}")
if len(errores):
    columnas = 6
    filas_fig = int(np.ceil(len(errores) / columnas))
    fig, axes = plt.subplots(filas_fig, columnas, figsize=(2.3 * columnas, 2.7 * filas_fig), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    rutas = prueba.assign(id=prueba["split"] + "/" + prueba["clase"] + "/" + prueba["archivo"]).set_index("id")["ruta"]
    for ax, (_, e) in zip(axes.ravel(), errores.iterrows()):
        with Image.open(rutas[e["id"]]) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(f"real: {e['clase_real']}\npred: {e['prediccion']} ({100 * e['confianza']:.0f} %)", fontsize=7)
    fig.suptitle("Errores con mayor confianza")
    fig.tight_layout()
    fig.savefig(CARPETA_RESULTADOS / "errores.png", dpi=150, bbox_inches="tight")
    plt.show()

## Umbral de Confianza

**Qué hace el chunk.** En la aplicación, la señal solo se muestra si la probabilidad de la clase ganadora supera un umbral; si no, se le pide al usuario acercar la cámara. Este chunk calcula, para umbrales de 0 a 0,95:
- **cobertura**: porcentaje de imágenes en las que el modelo sí da una respuesta;
- **accuracy** sobre esas imágenes.

Sirve para elegir el umbral por defecto de la aplicación (un equilibrio entre responder casi siempre y equivocarse poco). Se guarda en `umbral_confianza.csv`.

**Salida.** *(Completar con los resultados al ejecutar.)*

In [ ]:
umbrales = np.round(np.arange(0, 0.96, 0.05), 2)
filas_umbral = []
for u in umbrales:
    cubiertas = resultado["confianza"] >= u
    filas_umbral.append({"umbral": u, "cobertura_pct": 100 * cubiertas.mean(),
                         "accuracy_pct": 100 * resultado.loc[cubiertas, "acierto"].mean() if cubiertas.any() else np.nan})
tabla_umbral = pd.DataFrame(filas_umbral).set_index("umbral").round(1)
display(guardar(tabla_umbral, "umbral_confianza", CARPETA_RESULTADOS).T)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tabla_umbral.index, tabla_umbral["cobertura_pct"], color="#2a78d6", linewidth=2, label="Cobertura")
ax.plot(tabla_umbral.index, tabla_umbral["accuracy_pct"], color="#eb6834", linewidth=2, label="Accuracy (imágenes cubiertas)")
ax.set_xlabel("Umbral de confianza")
ax.set_ylabel("%")
ax.set_title("Cobertura y accuracy según el umbral de confianza")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(CARPETA_RESULTADOS / "umbral_confianza.png", dpi=150, bbox_inches="tight")
plt.show()

# Experimento: Robustez ante el Daltonismo

**Pregunta.** ¿Cuánto se pierde para reconocer una señal cuando la imagen se ve como la percibe una persona con daltonismo, y cuánto se recupera con la corrección de la aplicación?

**Diseño.** El modelo se usa como un "observador" estandarizado. Sobre una muestra del conjunto de prueba se comparan siete condiciones:

| Condición | Qué recibe el modelo |
| --- | --- |
| `original` | La imagen tal como es (visión típica). |
| `<tipo>_sin_correccion` | La imagen **como la percibe** una persona con ese tipo de daltonismo: `simular(imagen)`. |
| `<tipo>_con_correccion` | La imagen corregida por la aplicación, **como la percibe** la persona: `simular(corregir(imagen))`. |

Si la corrección funciona, el desempeño en `con_correccion` debería acercarse más al de `original` que el de `sin_correccion`.

**Limitaciones.** El modelo no es una persona: reconoce las señales principalmente por la forma y puede no depender tanto del color como un conductor. Por eso el experimento se complementa con la pérdida de color (ΔE) medida en los descriptivos. Además, la simulación es un modelo matemático de la dicromacia, no una medición con usuarios reales.

**Costo.** Con `N_EXPERIMENTO = 20` imágenes por clase son 200 imágenes × 6 condiciones nuevas = **1.200 llamadas** al servicio (la condición `original` se toma de la caché de la evaluación). Todas quedan en caché.

In [ ]:
N_EXPERIMENTO = 20     # imágenes por clase

experimento = pd.concat([g.sample(min(len(g), N_EXPERIMENTO), random_state=0)
                         for _, g in prueba.groupby("clase")], ignore_index=True)

CONDICIONES = {"original": None}
for t in TIPOS:
    CONDICIONES[f"{t}_sin_correccion"] = (lambda a, t=t: dal.simular(a, t))
    CONDICIONES[f"{t}_con_correccion"] = (lambda a, t=t: dal.simular(dal.corregir(a, t), t))

print(f"Muestra del experimento: {len(experimento)} imágenes ({N_EXPERIMENTO} por clase)")
pred_experimento = {}
for condicion, transformar in CONDICIONES.items():
    pred_experimento[condicion] = predecir_conjunto(experimento, condicion, transformar)
print("Listo.")

## Resultados del Experimento

**Qué hace el chunk.** Para cada condición calcula accuracy, F1 macro, AUC macro y error de categoría. Los guarda en `resultados_experimento.csv` y los grafica (`experimento_daltonismo.png`): una barra sin corrección y otra con corrección por cada tipo de daltonismo, con una línea en el valor de la condición original.

**Salida.** *(Completar con los resultados al ejecutar.)* Comparar, para cada tipo, la caída frente a `original` sin corrección y cuánto de esa caída se recupera con la corrección. Si la caída es pequeña incluso sin corrección, el modelo se apoya sobre todo en la forma; eso no invalida la corrección, porque la persona sí depende del color para la categoría (ver la tabla de ΔE).

In [ ]:
filas_exp = []
for condicion, pred in pred_experimento.items():
    if not len(pred):
        continue
    yr, yp, Pc = separar(pred)
    Yc = label_binarize(yr, classes=clases)
    presentes = Yc.sum(axis=0) > 0
    tipo = next((t for t in TIPOS if condicion.startswith(t)), "ninguno")
    filas_exp.append({
        "condicion": condicion, "tipo": tipo,
        "correccion": "con" if condicion.endswith("con_correccion") else ("sin" if tipo != "ninguno" else "-"),
        "imagenes": len(pred),
        "accuracy": accuracy_score(yr, yp),
        "f1_macro": f1_score(yr, yp, labels=clases, average="macro", zero_division=0),
        "auc_macro": roc_auc_score(Yc[:, presentes], Pc[:, presentes], average="macro"),
        "error_categoria_pct": 100 * (categoria_de(yr) != categoria_de(yp)).mean(),
        "confianza_media": Pc.max(axis=1).mean(),
    })
resultados_exp = pd.DataFrame(filas_exp).set_index("condicion").round(4)
display(guardar(resultados_exp, "resultados_experimento", CARPETA_RESULTADOS))

base = resultados_exp.loc["original"] if "original" in resultados_exp.index else None
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metrica, titulo in [(axes[0], "accuracy", "Accuracy"), (axes[1], "f1_macro", "F1 macro"),
                            (axes[2], "error_categoria_pct", "Error de categoría (%)")]:
    x = np.arange(len(TIPOS))
    for k, (corr, color) in enumerate([("sin", "#bdbdbd"), ("con", "#1baf7a")]):
        valores = [resultados_exp.loc[f"{t}_{corr}_correccion", metrica]
                   if f"{t}_{corr}_correccion" in resultados_exp.index else np.nan for t in TIPOS]
        barras = ax.bar(x + (k - 0.5) * 0.38, valores, width=0.38, color=color, label=f"{corr} corrección")
        ax.bar_label(barras, fmt="%.2f" if metrica != "error_categoria_pct" else "%.1f", fontsize=8, padding=2)
    if base is not None:
        ax.axhline(base[metrica], color="#2a78d6", linestyle="--", linewidth=1.5, label="original")
    ax.set_xticks(x, [dal.TIPOS[t].split(" ")[0] for t in TIPOS])
    ax.set_title(titulo)
    ax.grid(axis="x", visible=False)
axes[0].legend(frameon=False, fontsize=8, loc="lower left")
fig.suptitle("Desempeño del modelo con la imagen como la percibe una persona con daltonismo")
fig.tight_layout()
fig.savefig(CARPETA_RESULTADOS / "experimento_daltonismo.png", dpi=150, bbox_inches="tight")
plt.show()

# Prueba la Aplicación

Esta sección permite que **cualquier persona que lea el notebook pruebe el modelo**. El modelo reconoce 10 señales de tránsito; si se le muestra otra cosa, igual elegirá la más parecida, porque un clasificador multiclase siempre reparte la probabilidad entre las clases que conoce. Funciona mejor con la señal **recortada y centrada**, como las imágenes de entrenamiento.

**Opción 1 — Con la URL de una imagen.** Al ejecutar la celda aparece un cuadro de texto. Pegue la dirección de una imagen (clic derecho sobre la imagen > *Copiar dirección de la imagen*) y presione **Enter**. Se muestran la imagen, la predicción, la categoría y la imagen corregida para cada tipo de daltonismo. Para terminar, deje el cuadro vacío y presione Enter.

**Opción 2 — Con una imagen del equipo.** La celda siguiente abre un botón para subir uno o varios archivos.

In [ ]:
display(Markdown("**Modelo:** VisionVial (Azure Custom Vision) · **Clases:** " + ", ".join(clases)))
display(HTML(
    "<small>¿Necesita imágenes de prueba? Busque en Wikimedia Commons: "
    "<a href='https://commons.wikimedia.org/wiki/Category:Road_signs_in_Colombia' target='_blank'>señales de Colombia</a>. "
    "Abra la imagen, haga clic derecho y elija <i>Copiar dirección de la imagen</i>.</small>"))


def mostrar_correccion(imagen):
    arr = np.asarray(imagen.convert("RGB"))
    fig, axes = plt.subplots(1, 1 + len(TIPOS), figsize=(3.2 * (1 + len(TIPOS)), 3.2))
    axes[0].imshow(arr)
    axes[0].set_title("Original", fontsize=9)
    for ax, t in zip(axes[1:], TIPOS):
        ax.imshow(dal.corregir(arr, t))
        ax.set_title(f"Corregida para\n{dal.TIPOS[t].lower()}", fontsize=9)
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    plt.show()


def clasificar_url(url):
    imagen = None
    try:  # descarga la imagen solo para mostrarla; la predicción la hace Azure con la URL
        r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        imagen = Image.open(io.BytesIO(r.content)).convert("RGB")
    except Exception:
        pass
    respuesta = predecir_url(url)
    mostrar_prediccion(respuesta, imagen, titulo=url if len(url) < 70 else url[:67] + "...")
    if imagen is not None:
        mostrar_correccion(imagen)


while True:
    url = input("\nPegue la URL de la imagen y presione Enter (deje vacío para terminar): ").strip()
    if not url or url.lower() in ("salir", "fin", "exit"):
        print("Prueba finalizada. Ejecute de nuevo la celda para probar más imágenes.")
        break
    if not url.lower().startswith("http"):
        print("La URL debe empezar por http:// o https://. Intente de nuevo.")
        continue
    print("Consultando el modelo...")
    try:
        clasificar_url(url)
    except Exception as e:
        print(f"No fue posible clasificar la imagen. {e}")
        print("Verifique que la URL apunte directamente a un archivo de imagen público (.jpg, .png).")

In [ ]:
# Opción 2: clasificar imágenes del propio equipo
ARCHIVOS_LOCALES = []   # fuera de Colab: escriba aquí las rutas, ej. ["mi_senal.jpg"]

if EN_COLAB:
    from google.colab import files
    print("Seleccione una o varias imágenes (jpg, png o bmp):")
    subidos = files.upload()                      # {nombre: bytes}
else:
    subidos = {Path(p).name: Path(p).read_bytes() for p in ARCHIVOS_LOCALES}

if not subidos:
    print("No se seleccionó ninguna imagen.")

for nombre, contenido in subidos.items():
    try:
        imagen = ImageOps.exif_transpose(Image.open(io.BytesIO(contenido))).convert("RGB")
        respuesta = predecir_imagen(imagen)
    except Exception as e:
        print(f"{nombre}: no fue posible clasificarla. {e}")
        continue
    mostrar_prediccion(respuesta, imagen, titulo=nombre)
    mostrar_correccion(imagen)

## Aplicación Web con Cámara (Streamlit)

La aplicación **VisionVial** lleva el modelo al celular. Es `app.py`, en la raíz del repositorio, y se publica en Streamlit Community Cloud, que da una URL con HTTPS (necesaria para que el navegador permita usar la cámara).

**Enlace de la aplicación:** *(pegar aquí la URL `https://<app>.streamlit.app`)*

| Pestaña | Qué hace |
| --- | --- |
| **📷 Foto** | Toma una foto con la cámara o sube una imagen. Envía la imagen **original** al endpoint de Custom Vision y muestra la señal, su categoría y su color escritos en texto y con una muestra de color. Si la confianza no supera el umbral, pide acercar la cámara. Debajo muestra la imagen **corregida** para el tipo de daltonismo elegido y, para personas sin daltonismo, una simulación de cómo la percibe el usuario antes y después de la corrección. Permite descargar la imagen corregida. |
| **🗂️ Lote / dataset** | Recibe varias imágenes o un **ZIP**. Si el ZIP tiene una carpeta por etiqueta (como `test/`), compara contra la clase real y muestra accuracy y matriz de confusión. Los resultados se descargan en CSV. |
| **👁️ Simulador** | Muestra una imagen como la percibe cada tipo de daltonismo, sin y con corrección. |
| **ℹ️ Acerca de** | Explica el funcionamiento y lista las señales que reconoce el modelo. |

En la barra lateral el usuario elige su **tipo de daltonismo**, la **severidad**, la **intensidad de la corrección** y el **umbral de confianza** (por defecto 0,6; ajustar según la sección *Umbral de Confianza*).

Además de usar el modelo, la aplicación muestra las métricas de `python vv.py evaluar` (pestaña *Desempeño del modelo*) y los pantallazos de Custom Vision (pestaña *Entrenamiento en ACV*). Como lee esos archivos del repositorio, se actualiza sola cada vez que se ejecuta `python vv.py publicar`.

**Qué hace el chunk.** Muestra las capturas de la aplicación importadas con `python vv.py capturas --origen <carpeta> --app`.

In [ ]:
mostrar_capturas(CARPETA_SNIP_APP)

# Conclusiones

*(Completar con los resultados obtenidos.)*

Con respecto a los datos.

  - *Balance de clases y categorías, tamaño original de los recortes de GTSRB y efecto de la ampliación.*
  - *Presencia de color por clase y diferencia entre la señalización alemana y la colombiana (preventivas).*
  - *Pérdida de color (ΔE) por tipo de daltonismo: qué clases y tipos son los más afectados.*

Con respecto al modelo en Azure Custom Vision.

  - *Métricas del portal frente a las del conjunto de prueba independiente (precision, recall, F1, AUC).*
  - *Clases que más se confunden y si los errores cruzan de categoría.*
  - *Umbral de confianza elegido para la aplicación.*

Con respecto al daltonismo y la aplicación.

  - *Resultado del experimento: caída sin corrección y recuperación con corrección por tipo de daltonismo.*
  - *Aporte de la aplicación: además de corregir el color, comunica la categoría en texto, lo que no depende de la percepción del color.*

Trabajo futuro.

  - Detección de señales en la escena completa (por ejemplo, con un modelo de detección de objetos de Custom Vision o YOLO) para no depender del encuadre manual.
  - Dataset de señales colombianas (rombos amarillos) y estado del semáforo (rojo, amarillo y verde).
  - Procesamiento de video en tiempo real y empaquetado como aplicación móvil.
  - Validación con personas con daltonismo.

# Referencias

- Fidaner, O., Lin, P. y Ozguven, N. (2005). *Analysis of Color Blindness*. Stanford University.
- Machado, G. M., Oliveira, M. M. y Fernandes, L. A. F. (2009). A physiologically-based model for simulation of color vision deficiency. *IEEE Transactions on Visualization and Computer Graphics*, 15(6), 1291–1298.
- Microsoft. *Azure AI Custom Vision documentation*. https://learn.microsoft.com/azure/ai-services/custom-vision-service/
- Ministerio de Transporte de Colombia. *Manual de Señalización Vial*.
- Stallkamp, J., Schlipsing, M., Salmen, J. e Igel, C. (2012). Man vs. computer: Benchmarking machine learning algorithms for traffic sign recognition. *Neural Networks*, 32, 323–332.